# 0. Setup: Install Requirements & Compile Marlin Kernel

Install Python dependencies and load the GPTQModel Marlin extension used later for fast 4-bit inference.

In [ ]:
# install the requirements
!pip install -r ../requirements.txt --no-build-isolation

In [ ]:
import gptqmodel; gptqmodel.extension.load("marlin")

# 1. Dataset Prep

Load the CORD-v2 receipt dataset (train/validation/test splits) from Hugging Face.

In [1]:
from datasets import load_dataset

# Load CORD-v2 receipt dataset (all 3 splits)
dataset = load_dataset("naver-clova-ix/cord-v2")

train_dataset = dataset["train"]
validation_dataset = dataset["validation"]
test_dataset = dataset["test"]

print(f"Train samples:      {len(train_dataset)}")
print(f"Validation samples: {len(validation_dataset)}")
print(f"Test samples:       {len(test_dataset)}")
print(f"\nFeatures: {train_dataset.features}")


Train samples:      800
Validation samples: 100
Test samples:       100

Features: {'image': Image(mode=None, decode=True), 'ground_truth': Value('string')}


Visualize a few random training samples (image + `gt_parse` JSON) to sanity-check the data.

In [ ]:
import random
import json
import base64
from io import BytesIO
from IPython.display import display, HTML

def show_random_samples(dataset, n=4):
    indices = random.sample(range(len(dataset)), n)

    html_parts = ['<div style="display:flex; flex-wrap:wrap; gap:16px;">']

    for idx in indices:
        sample = dataset[idx]
        gt = json.loads(sample["ground_truth"])

        # Keep only gt_parse
        gt_parse = gt.get("gt_parse", {})

        # Convert PIL image to base64 PNG
        buf = BytesIO()
        sample["image"].save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode()

        json_str = json.dumps(gt_parse, indent=2)

        html_parts.append(f"""
        <div style="border:1px solid #ccc; padding:10px; border-radius:8px; max-width:340px;">
            <p style="font-weight:bold; margin:0 0 6px 0;">Sample #{idx}</p>
            <img src="data:image/png;base64,{b64}"
                 style="max-width:300px; width:100%; display:block; border-radius:4px;"/>
            <pre style="font-size:9px; background:#f5f5f5; padding:8px; border-radius:4px;
                        max-height:220px; overflow-y:auto; white-space:pre-wrap;
                        margin-top:8px; font-family:monospace;">{json_str}</pre>
        </div>
        """)

    html_parts.append("</div>")
    display(HTML("".join(html_parts)))


In [ ]:
show_random_samples(train_dataset, n=3)

Extract the `gt_parse` object from each row and **losslessly** normalize its label structure: single-item `menu`/`sub` containers are wrapped into lists, and leaves that appear as both a string and a list (rule computed from train+val only, frozen to `mixed_fields.json`) are coerced to list-of-string. Returns three lazy splits whose images decode on access. Nothing is stripped, so receipts with keys outside the schema are reported as rejects below — not silently dropped.

In [2]:
from utils import preprocess_cord
train_dataset, validation_dataset, test_dataset = preprocess_cord(dataset)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.8.0+cu128).


mixed fields (12) frozen to mixed_fields.json
sizes: train=800 val=100 test=100
[train] reject: 1 validation error for Receipt
void_menu
  Extra inputs are not permitted [type=extra_forbidden, input_value={'nm': 'SOP AYM BNG', 'price': '-7,000'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
  label={"menu": [{"nm": ["SOP AYM BNG"], "price": "7,000"}, {"nm": ["TEH TARIK P"], "price": "15,000"}], "total": {"total_price": ["15,000"], "cashprice": ["55,000"], "changeprice": ["40,000"]}, "void_menu":
[train] reject: 1 validation error for Receipt
sub_total.othersvc_price
  Extra inputs are not permitted [type=extra_forbidden, input_value=['27,300', '0'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
  label={"menu": [{"nm": ["KUE CUBIT OVO/ SKIPPY"], "cnt": ["1"], "price": "39,000"}, {"nm": ["ES BUAH"], "cnt": ["2"], "price": "36,000"}, {"nm": ["DODOT KAKEK"], "cnt": ["1"], "pric

# 2. Model Prep & Fine-Tuning

Download the base model and set the fine-tuning hyperparameters (LoRA rank, tuned modules, max pixels).

In [ ]:
from huggingface_hub import snapshot_download
from utils import BASE_MODEL_SAVE_DIR

MODEL_ID_REPO = "Qwen/Qwen3-VL-4B-Instruct"
MODEL_NAME = "qwen3vl-4b"
MODEL_MAX_PX = 1600
MODEL_LORA_RANK = 128
MODEL_LT_TUNE = True
MODEL_VT_TUNE = False

MODEL_ID = snapshot_download(
    repo_id=MODEL_ID_REPO,
    cache_dir=BASE_MODEL_SAVE_DIR + MODEL_NAME,
)
print("downloaded to", MODEL_ID)

Load the base model and processor into memory.

In [ ]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="flash_attention_2",
)
processor = AutoProcessor.from_pretrained(MODEL_ID, max_pixels=MODEL_MAX_PX*28*28)

# REQUIRED for the prefix-based prompt masking below to land on the right tokens
processor.tokenizer.padding_side = "right"

Inspect the model's linear layers to identify LoRA target module names, for both the vision tower and the language model.

In [ ]:
import re
seen = set()
for name, mod in model.named_modules():
    if mod.__class__.__name__ == "Linear" and (".visual" in name or "vision" in name):
        key = re.sub(r"\.\d+\.", ".N.", name)
        if key not in seen:
            seen.add(key)
            print(name)

In [ ]:
for name, mod in model.named_modules():
    if mod.__class__.__name__ == "Linear":
        tower = "VISION" if (".visual" in name or "vision" in name) else "LANG  "
        print(f"[{tower}] {name}")

Configure the LoRA adapter (target modules, rank) and derive this run's output naming convention.

In [ ]:
from peft import LoraConfig

LM_TARGETS     = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]
VISION_TARGETS = ["qkv", "proj", "linear_fc1", "linear_fc2"]

training_targets = []
if MODEL_LT_TUNE:
    training_targets.extend(LM_TARGETS)
if MODEL_VT_TUNE:
    training_targets.extend(VISION_TARGETS)

peft_config = LoraConfig(
    r=MODEL_LORA_RANK,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=training_targets,
)

ft_targets = ""
if MODEL_LT_TUNE:
    ft_targets += "LT"
if MODEL_VT_TUNE:
    ft_targets += "VT"
full_model_name = f"{MODEL_NAME}-r{MODEL_LORA_RANK}-{ft_targets}-px{MODEL_MAX_PX}"

Define the data collator: builds prompt+label chat text, tokenizes it, and masks the prompt/image/padding tokens out of the training loss.

In [ ]:
from utils import FIXED_PROMPT

def collate_fn(examples):
    """Build full (prompt+label) and prompt-only chat texts per example, for prefix masking below."""
    full_texts, prompt_texts, images = [], [], []
    for ex in examples:
        user_turn = [{"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": FIXED_PROMPT},
        ]}]
        full_msg = user_turn + [{"role": "assistant", "content": [
            {"type": "text", "text": json.dumps(ex["label"], ensure_ascii=False)},
        ]}]
        full_texts.append(
            processor.apply_chat_template(full_msg, tokenize=False, add_generation_prompt=False))
        prompt_texts.append(
            processor.apply_chat_template(user_turn, tokenize=False, add_generation_prompt=True))
        images.append(ex["image"])

    batch = processor(text=full_texts, images=images, padding=True, return_tensors="pt")
    labels = batch["input_ids"].clone()

    # mask the fixed prompt + image tokens per sample (right padding => prompt at the front);
    # re-run the processor on prompt-only WITH the image so image-token expansion is counted
    for i, (p_text, img) in enumerate(zip(prompt_texts, images)):
        plen = processor(text=[p_text], images=[img], return_tensors="pt")["input_ids"].shape[1]
        labels[i, :plen] = -100

    labels[labels == processor.tokenizer.pad_token_id] = -100   # mask padding

    # defensively mask any image placeholder tokens that survive in the completion region
    image_token_id = getattr(model.config, "image_token_id", None)
    if image_token_id is not None:
        labels[labels == image_token_id] = -100

    batch["labels"] = labels
    return batch

Configure training arguments and build the `SFTTrainer`.

In [ ]:
from trl import SFTConfig, SFTTrainer
from utils import CHECKPOINT_MODEL_SAVE_DIR

training_args = SFTConfig(
    output_dir=CHECKPOINT_MODEL_SAVE_DIR + full_model_name,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=5,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_steps=50,
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    eval_strategy="steps",
    eval_steps=50,
    per_device_eval_batch_size=2,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=collate_fn,
    peft_config=peft_config,    
    processing_class=processor, 
)

Run fine-tuning and save the resulting LoRA adapter.

In [ ]:
from utils import ADAPTER_OUTPUT_DIR

trainer.train()
trainer.save_model(ADAPTER_OUTPUT_DIR + full_model_name)

Free the trainer and model from memory, releasing GPU VRAM before evaluation.

In [ ]:
trainer.accelerator.free_memory()

trainer.model = None
trainer.model_wrapped = None
trainer.optimizer = None
trainer.lr_scheduler = None

del trainer
del model

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:5.2f} GB | "
          f"reserved: {torch.cuda.memory_reserved()/1e9:5.2f} GB")

# 3. Model Evaluation

Discover saved checkpoints for this run (base model + each checkpoint + the final adapter) to evaluate.

In [ ]:
import glob
import os
import re

from utils import select_best, CHECKPOINT_MODEL_SAVE_DIR, ADAPTER_OUTPUT_DIR

def checkpoint_sort_key(path):
    match = re.search(r"checkpoint-(\d+)$", os.path.normpath(path))
    return int(match.group(1)) if match else -1

checkpoint_dir = os.path.join(CHECKPOINT_MODEL_SAVE_DIR, full_model_name)
checkpoints = sorted(
    glob.glob(os.path.join(checkpoint_dir, "checkpoint-*")),
    key=checkpoint_sort_key,
)
if checkpoints:
    print(f"Skipping last checkpoint (redundant with final adapter): {checkpoints[-1]}")
    checkpoints = checkpoints[:-1]

adapter_path = os.path.join(ADAPTER_OUTPUT_DIR, full_model_name)
adapter_paths = [None] + checkpoints + [adapter_path]
print(f"Evaluating base model + {len(checkpoints)} checkpoint(s) + final adapter")

Rank the base model and all checkpoints by field-F1 on the validation set, using plain-text decoding.

In [ ]:
ranking = select_best(
    base_path=MODEL_ID,
    adapter_paths=adapter_paths,
    select_ds=validation_dataset,
    decode="plain",
    max_pixels=MODEL_MAX_PX * 28 * 28,
)
best_path = ranking[0]["path"]

Repeat the ranking using schema-constrained decoding.

In [ ]:
ranking = select_best(
    base_path=MODEL_ID,
    adapter_paths=adapter_paths,
    select_ds=validation_dataset,
    decode="constrained",
    max_pixels=MODEL_MAX_PX * 28 * 28,
)
best_path = ranking[0]["path"]

# 4. Merge Adapter

Merge the chosen LoRA adapter into the base model and save the merged model to disk.

In [ ]:
import os

from utils import load_model_for_inference, MERGED_MODEL_SAVE_DIR

# Path to the adapter/checkpoint folder to merge into its base model.
ADAPTER_PATH = "../models/checkpoints/qwen3vl-4b-r16-LT-px1600/checkpoint-900"
# Folder name the merged model will be saved under, i.e. MERGED_MODEL_SAVE_DIR/MERGE_MODEL_NAME
MERGE_MODEL_NAME = "qwen3vl-4b-r16-LT-px1600-checkpoint-900"

merged_model, merged_processor = load_model_for_inference(
    base_path=MODEL_ID,
    adapter_path=ADAPTER_PATH,
    merge=True,
    max_pixels=MODEL_MAX_PX * 28 * 28,
)

merged_save_path = os.path.join(MERGED_MODEL_SAVE_DIR, MERGE_MODEL_NAME)
merged_model.save_pretrained(merged_save_path)
merged_processor.save_pretrained(merged_save_path)
print(f"Saved merged model to {merged_save_path}")

Reload the merged model from disk and evaluate it on the held-out test split.

In [ ]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from utils import run_inference

merged_test_model = Qwen3VLForConditionalGeneration.from_pretrained(
    merged_save_path,
    dtype=torch.bfloat16,
    device_map="auto",
)
merged_test_processor = AutoProcessor.from_pretrained(merged_save_path, max_pixels=MODEL_MAX_PX * 28 * 28)

metrics, preds, labels = run_inference(merged_test_model, merged_test_processor, test_dataset)
print(f"field_f1={metrics['field_f1']:.4f}  normalized_ted={metrics['normalized_ted']:.4f}  "
      f"json_validity={metrics['json_validity']:.4f}  exact_match={metrics['exact_match']:.4f}")

# 5. Quantization

Point at the merged model on disk by name (not the in-memory model from section 4). Only the processor is loaded here — `GPTQModel.load()` below reads the weights from disk itself.

In [ ]:
import os

import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from utils import MERGED_MODEL_SAVE_DIR, FIXED_PROMPT

# Resolved strictly from disk by name, not reused from the merge cell above.
QUANT_MERGE_MODEL_NAME = "qwen3vl-4b-r16-LT-px1600-checkpoint-900"
QUANT_SOURCE_PATH = os.path.join(MERGED_MODEL_SAVE_DIR, QUANT_MERGE_MODEL_NAME)

quant_source_processor = AutoProcessor.from_pretrained(QUANT_SOURCE_PATH, max_pixels=MODEL_MAX_PX * 28 * 28)
print(f"Quantization source: {QUANT_SOURCE_PATH}")

**Test 1: GPTQ (4-bit, calibration-based, via GPTQModel)**

Quantizes only the language-model decoder linears (vision tower, embeddings, `lm_head` excluded). Calibration is multimodal: each sample is a full chat conversation with the receipt image, fixed prompt and gold JSON answer.

In [ ]:
import json
import random

from gptqmodel import GPTQModel, QuantizeConfig

GPTQ_BITS = 4
GPTQ_GROUP_SIZE = 128
GPTQ_NUM_CALIBRATION_SAMPLES = 256

calib_indices = random.sample(range(len(train_dataset)), GPTQ_NUM_CALIBRATION_SAMPLES)
gptq_calibration = []
for idx in calib_indices:
    ex = train_dataset[idx]
    gptq_calibration.append([
        {"role": "user", "content": [
            {"type": "image", "image": ex["image"]},
            {"type": "text", "text": FIXED_PROMPT},
        ]},
        {"role": "assistant", "content": [
            {"type": "text", "text": json.dumps(ex["label"], ensure_ascii=False)},
        ]},
    ])

quant_config = QuantizeConfig(bits=GPTQ_BITS, group_size=GPTQ_GROUP_SIZE)

# Loads its own copy of the merged weights; layers stream from disk during
# quantization (offload_to_disk defaults to True), keeping peak VRAM low.
gptq_model = GPTQModel.load(QUANT_SOURCE_PATH, quant_config)
gptq_model.quantize(gptq_calibration, batch_size=1)

Save the GPTQ-quantized model and free GPU memory.

In [ ]:
import gc

from utils import QUANTIZED_MODEL_SAVE_DIR

gptq_save_path = os.path.join(QUANTIZED_MODEL_SAVE_DIR, f"{QUANT_MERGE_MODEL_NAME}-gptq")
gptq_model.save(gptq_save_path)
# re-save the processor loaded with max_pixels so inference-time preprocessing matches
quant_source_processor.save_pretrained(gptq_save_path)
print(f"Saved GPTQ-quantized model to {gptq_save_path}")

del gptq_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:5.2f} GB | "
          f"reserved: {torch.cuda.memory_reserved()/1e9:5.2f} GB")

**Test 2: bitsandbytes NF4** (4-bit, no calibration — language model only; vision tower kept in bf16 to match GPTQ).

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual", "lm_head"],
)

bnb_model = Qwen3VLForConditionalGeneration.from_pretrained(
    QUANT_SOURCE_PATH,
    quantization_config=bnb_config,
    device_map="auto",
)

Save the bitsandbytes NF4 model and free GPU memory.

In [ ]:
import gc

from utils import QUANTIZED_MODEL_SAVE_DIR

bnb_save_path = os.path.join(QUANTIZED_MODEL_SAVE_DIR, f"{QUANT_MERGE_MODEL_NAME}-bnb-nf4")
bnb_model.save_pretrained(bnb_save_path)
quant_source_processor.save_pretrained(bnb_save_path)
print(f"Saved bitsandbytes NF4 model to {bnb_save_path}")

del bnb_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:5.2f} GB | "
          f"reserved: {torch.cuda.memory_reserved()/1e9:5.2f} GB")

Compare the two quantized models on the test split.

In [ ]:
import os
import gc

from gptqmodel import GPTQModel
from utils import run_inference

from utils import QUANTIZED_MODEL_SAVE_DIR

gptq_save_path = os.path.join(QUANTIZED_MODEL_SAVE_DIR, f"{QUANT_MERGE_MODEL_NAME}-gptq")
bnb_save_path = os.path.join(QUANTIZED_MODEL_SAVE_DIR, f"{QUANT_MERGE_MODEL_NAME}-bnb-nf4")

def load_quantized(path, kind):
    if kind == "gptq":
        return GPTQModel.load(path, device_map="auto").model
    return Qwen3VLForConditionalGeneration.from_pretrained(path, device_map="auto")

for label, path, kind in [
    ("GPTQ (4-bit, calibrated)", gptq_save_path, "gptq"),
    ("bitsandbytes NF4", bnb_save_path, "bnb"),
]:
    q_model = load_quantized(path, kind)
    q_processor = AutoProcessor.from_pretrained(path, max_pixels=MODEL_MAX_PX * 28 * 28)
    metrics, _, _ = run_inference(q_model, q_processor, test_dataset)
    print(f"[{label}] field_f1={metrics['field_f1']:.4f}  normalized_ted={metrics['normalized_ted']:.4f}  "
          f"json_validity={metrics['json_validity']:.4f}  exact_match={metrics['exact_match']:.4f}")
    del q_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 6. Compare 4B GPTQ vs 2B Fine-Tune

Head-to-head on the test split: the GPTQ-quantized 4B merged model vs the (unquantized) 2B fine-tune at LoRA rank 16, checkpoint-1200. Both were trained at px1600.

**Self-contained**: only section 1 (Dataset Prep) needs to run first — everything else (paths, models, processors) is resolved here from disk.

## 6.1 Setup

Resolve model/adapter paths for both contenders and download the 2B base model.

In [3]:
import gc
import os

import torch
from huggingface_hub import snapshot_download
from utils import (BASE_MODEL_SAVE_DIR, CHECKPOINT_MODEL_SAVE_DIR,
                   QUANTIZED_MODEL_SAVE_DIR)

CMP_MAX_PX = 1600  # both contenders were trained at px1600

CMP_GPTQ_4B_PATH = os.path.join(
    QUANTIZED_MODEL_SAVE_DIR, "qwen3vl-4b-r16-LT-px1600-checkpoint-900-gptq")
CMP_2B_ADAPTER_PATH = os.path.join(
    CHECKPOINT_MODEL_SAVE_DIR, "qwen3vl-2b-r16-LT-px1600", "checkpoint-1200")
    # resolves from the local HF cache (already downloaded during 2B training)
CMP_2B_BASE_PATH = snapshot_download(
    repo_id="Qwen/Qwen3-VL-2B-Instruct",
    cache_dir=BASE_MODEL_SAVE_DIR + "qwen3vl-2b",
)

cmp_results = {}

def free_gpu(*names):
    """Drop globals by name and return CUDA memory to the driver."""
    g = globals()
    for n in names:
        g.pop(n, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:5.2f} GB | "
              f"reserved: {torch.cuda.memory_reserved()/1e9:5.2f} GB")

print(f"4B GPTQ:    {CMP_GPTQ_4B_PATH}")
print(f"2B base:    {CMP_2B_BASE_PATH}")
print(f"2B adapter: {CMP_2B_ADAPTER_PATH}")

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

4B GPTQ:    ../models/quantized/qwen3vl-4b-r16-LT-px1600-checkpoint-900-gptq
2B base:    /workspace/fine-tune-receipe-pdf/models/base/qwen3vl-2b/models--Qwen--Qwen3-VL-2B-Instruct/snapshots/89644892e4d85e24eaac8bacfd4f463576704203
2B adapter: ../models/checkpoints/qwen3vl-2b-r16-LT-px1600/checkpoint-1200


## 6.2 Contender 1 — 4B GPTQ (4-bit)

Evaluate the 4B merged model with GPTQ 4-bit quantization.

In [4]:
# --- contender 1: 4B merged + GPTQ 4-bit ---
from gptqmodel import GPTQModel
from transformers import AutoProcessor
from utils import run_inference

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# GPTQModel reloads its own checkpoints natively (plain transformers would need
# `optimum`); `.model` is the underlying HF model that run_inference expects
gptq4b_model = GPTQModel.load(CMP_GPTQ_4B_PATH, device_map="auto").model
gptq4b_processor = AutoProcessor.from_pretrained(CMP_GPTQ_4B_PATH, max_pixels=CMP_MAX_PX * 28 * 28)

gpu_model_gb = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0

metrics, _, _ = run_inference(gptq4b_model, gptq4b_processor, test_dataset)
metrics["gpu_model_gb"] = gpu_model_gb  # weights footprint right after load
metrics["gpu_peak_gb"] = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
cmp_results["4B GPTQ (ckpt-900, 4-bit)"] = metrics
print(metrics)

free_gpu("gptq4b_model", "gptq4b_processor")

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.1.0+656ac5f
Transformers : 5.13.0
Torch        : 2.8.0+cu128
Triton       : 3.4.0


from_quantized: adapter: None


...

INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.


INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.0,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}



INFO  Kernel: loaded -> `[MarlinLinear]`                                       


inference:   0%|          | 0/13 [00:00<?, ?batch/s]

{'n': 100, 'json_validity': 0.97, 'exact_match': 0.42, 'field_precision': 0.8942084942084942, 'field_recall': 0.8900845503458877, 'field_f1': 0.8921417565485362, 'normalized_ted': np.float64(0.9164693876316535), 'gpu_model_gb': 3.505121792, 'gpu_peak_gb': 6.207523328}
GPU allocated:  0.01 GB | reserved:  0.25 GB


## 6.3 Contender 2 — 2B fine-tune (bf16)

Evaluate the 2B LoRA fine-tune (bf16 base + adapter, unquantized).

In [5]:
# --- contender 2: 2B r16 checkpoint-1200 (bf16 base + LoRA adapter, unquantized) ---
from utils import load_model_for_inference, run_inference

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

model_2b, processor_2b = load_model_for_inference(
    base_path=CMP_2B_BASE_PATH,
    adapter_path=CMP_2B_ADAPTER_PATH,
    merge=False,
    max_pixels=CMP_MAX_PX * 28 * 28,
)

gpu_model_gb = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0

metrics, _, _ = run_inference(model_2b, processor_2b, test_dataset)
metrics["gpu_model_gb"] = gpu_model_gb  # weights footprint right after load
metrics["gpu_peak_gb"] = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
cmp_results["2B r16 (ckpt-1200, bf16)"] = metrics
print(metrics)

free_gpu("model_2b", "processor_2b")

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

inference:   0%|          | 0/13 [00:00<?, ?batch/s]

{'n': 100, 'json_validity': 0.96, 'exact_match': 0.43, 'field_precision': 0.8699311400153023, 'field_recall': 0.8739431206764028, 'field_f1': 0.8719325153374233, 'normalized_ted': np.float64(0.8937998600847311), 'gpu_model_gb': 4.334196224, 'gpu_peak_gb': 6.868583936}
GPU allocated:  0.01 GB | reserved:  0.25 GB


## 6.4 Side-by-side results

Print a side-by-side comparison of quality metrics and GPU memory usage.

In [6]:
# --- side-by-side summary ---
# (metric, format): quality scores + GPU memory (weights footprint after load, peak during inference)
METRIC_COLS = [
    ("field_f1", ".4f"), ("normalized_ted", ".4f"),
    ("json_validity", ".4f"), ("exact_match", ".4f"),
    ("gpu_model_gb", ".2f"), ("gpu_peak_gb", ".2f"),
]

header = f"{'model':<28}" + "".join(f"{k:>16}" for k, _ in METRIC_COLS)
print(header)
print("-" * len(header))
for name, m in cmp_results.items():
    print(f"{name:<28}" + "".join(f"{m.get(k, float('nan')):>16{fmt}}" for k, fmt in METRIC_COLS))

if len(cmp_results) == 2:
    (name_a, m_a), (name_b, m_b) = cmp_results.items()
    print("-" * len(header))
    print(f"{'delta (first - second)':<28}"
          + "".join(f"{m_a.get(k, float('nan')) - m_b.get(k, float('nan')):>+16{fmt}}"
                    for k, fmt in METRIC_COLS))

model                               field_f1  normalized_ted   json_validity     exact_match    gpu_model_gb     gpu_peak_gb
----------------------------------------------------------------------------------------------------------------------------
4B GPTQ (ckpt-900, 4-bit)             0.8921          0.9165          0.9700          0.4200            3.51            6.21
2B r16 (ckpt-1200, bf16)              0.8719          0.8938          0.9600          0.4300            4.33            6.87
----------------------------------------------------------------------------------------------------------------------------
delta (first - second)               +0.0202         +0.0227         +0.0100         -0.0100           -0.83           -0.66


# 7. Export & Publish the Winning Model

Two full-model export formats (both include the vision tower), then publishing to the Hugging Face Hub. Outputs go to `models/export/`.

ONNX was considered and dropped: no ONNX toolchain exports the full Qwen3-VL pipeline.

## 7.1 Export option 1: GGUF (llama.cpp)

Full model (LM + vision). Converts the merged bf16 checkpoint (not the GPTQ one) into an F16 GGUF plus a companion `mmproj` vision file, runnable via `llama-server`, Ollama or LM Studio.

In [ ]:
import os

from utils import MERGED_MODEL_SAVE_DIR, EXPORT_MODEL_SAVE_DIR

GGUF_SRC_NAME = "qwen3vl-4b-r16-LT-px1600-checkpoint-900"
GGUF_SRC_PATH = os.path.join(MERGED_MODEL_SAVE_DIR, GGUF_SRC_NAME)
GGUF_OUT_PATH = os.path.join(EXPORT_MODEL_SAVE_DIR, f"{GGUF_SRC_NAME}-gguf")
LLAMA_CPP_DIR = os.path.join(EXPORT_MODEL_SAVE_DIR, "_tools", "llama.cpp")

os.makedirs(GGUF_OUT_PATH, exist_ok=True)

# the converter runs from the repo checkout (it vendors its own gguf-py, no pip deps needed)
if not os.path.exists(LLAMA_CPP_DIR):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_CPP_DIR}

# 1) language model -> F16 GGUF
!python {LLAMA_CPP_DIR}/convert_hf_to_gguf.py {GGUF_SRC_PATH} \
    --outfile {GGUF_OUT_PATH}/{GGUF_SRC_NAME}-f16.gguf --outtype f16

# 2) vision tower -> mmproj GGUF (kept F16; quantizing the projector is not recommended)
!python {LLAMA_CPP_DIR}/convert_hf_to_gguf.py {GGUF_SRC_PATH} \
    --outfile {GGUF_OUT_PATH}/{GGUF_SRC_NAME}-f16.gguf --outtype f16 --mmproj

for f in sorted(os.listdir(GGUF_OUT_PATH)):
    print(f"{os.path.getsize(os.path.join(GGUF_OUT_PATH, f))/1e9:6.2f} GB  {f}")

Optional: further quantize the F16 GGUF to Q4_K_M (builds the `llama-quantize` binary once via cmake).

In [ ]:
# OPTIONAL: 4-bit GGUF (Q4_K_M). Needs the llama-quantize binary -> one-time cmake build (~5 min, CPU-only).
LLAMA_QUANTIZE = os.path.join(LLAMA_CPP_DIR, "build", "bin", "llama-quantize")

if not os.path.exists(LLAMA_QUANTIZE):
    !cmake -S {LLAMA_CPP_DIR} -B {LLAMA_CPP_DIR}/build -DGGML_CUDA=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF > /dev/null
    !cmake --build {LLAMA_CPP_DIR}/build --target llama-quantize -j > /dev/null

gguf_f16 = os.path.join(GGUF_OUT_PATH, f"{GGUF_SRC_NAME}-f16.gguf")
gguf_q4  = os.path.join(GGUF_OUT_PATH, f"{GGUF_SRC_NAME}-Q4_K_M.gguf")

!{LLAMA_QUANTIZE} {gguf_f16} {gguf_q4} Q4_K_M

for f in sorted(os.listdir(GGUF_OUT_PATH)):
    print(f"{os.path.getsize(os.path.join(GGUF_OUT_PATH, f))/1e9:6.2f} GB  {f}")

## 7.2 Export option 2: OpenVINO IR (optimum-intel)

Full model, for Intel CPU/GPU/NPU targets (no CUDA backend). `--weight-format int4` applies NNCF data-free int4 weight compression.

In [ ]:
# installed on demand rather than via requirements.txt: pulls openvino + nncf and
# upgrades optimum to 2.x, which the rest of the notebook doesn't need
!pip install -q "optimum-intel[openvino]>=2.0"

OV_OUT_PATH = os.path.join(EXPORT_MODEL_SAVE_DIR, f"{GGUF_SRC_NAME}-openvino-int4")

!optimum-cli export openvino \
    --model {GGUF_SRC_PATH} \
    --task image-text-to-text \
    --weight-format int4 \
    {OV_OUT_PATH}

total = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(OV_OUT_PATH) for f in fs)
print(f"\nexport size: {total/1e9:.2f} GB in {OV_OUT_PATH}")
for f in sorted(os.listdir(OV_OUT_PATH)):
    print(" ", f)

## 7.3 Publish to the Hugging Face Hub

Publish the exported models as three separate private repos (GPTQ, GGUF, OpenVINO). Requires an `HF_TOKEN` write token set in the environment.

In [ ]:
from huggingface_hub import HfApi
from utils import QUANTIZED_MODEL_SAVE_DIR

HF_USER = "your-username"
HF_BASE_NAME = "qwen3vl-4b-receipt-extraction"

SAFETENSORS_PATH = os.path.join(QUANTIZED_MODEL_SAVE_DIR, "qwen3vl-4b-r16-LT-px1600-checkpoint-900-gptq")

uploads = {
    f"{HF_USER}/{HF_BASE_NAME}-gptq":     SAFETENSORS_PATH,  # GPTQ int4 safetensors (full multimodal model)
    f"{HF_USER}/{HF_BASE_NAME}-gguf":     GGUF_OUT_PATH,     # F16 + Q4_K_M GGUF + mmproj vision file
    f"{HF_USER}/{HF_BASE_NAME}-openvino": OV_OUT_PATH,       # OpenVINO IR int4 (full pipeline)
}

if not (os.environ.get("HF_TOKEN") or os.path.exists(os.path.expanduser("~/.cache/huggingface/token"))):
    print("No HF credentials found. Set the HF_TOKEN env var (write token) or run:")
    print("  from huggingface_hub import login; login()")
else:
    api = HfApi()
    for repo_id, folder in uploads.items():
        if not os.path.isdir(folder):
            print(f"SKIP {repo_id}: {folder} does not exist (run its export cell first)")
            continue
        api.create_repo(repo_id, private=True, exist_ok=True)
        api.upload_folder(folder_path=folder, repo_id=repo_id,
                          commit_message=f"upload from {os.path.basename(folder)}")
        print(f"Uploaded {folder} -> https://huggingface.co/{repo_id} (private)")